# 🌍 World Happiness Report 2026 — Complete EDA & Analysis

**147 countries · 6 happiness factors · Official WHR 2026 Rankings**

> *"Happiness is not something ready-made. It comes from your own actions."* — Dalai Lama

### What This Notebook Covers
1. Dataset Overview & Data Quality
2. Global Happiness Distribution
3. Regional Analysis
4. The Six Happiness Factors
5. Correlation Analysis
6. Top & Bottom Countries Deep-Dive
7. GDP vs Happiness — The Classic Question
8. Social Media & Youth Wellbeing (2026 Special Focus)
9. Geospatial Happiness Map
10. Regression Model — Predicting Happiness Score

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'sans-serif'

GOLD   = '#F4C430'
TEAL   = '#2EC4B6'
CORAL  = '#E84855'
PURPLE = '#9B5DE5'
GREEN  = '#06D6A0'
NAVY   = '#073B4C'

print("✅ Libraries loaded successfully")

## 1. Dataset Overview

In [ ]:
df = pd.read_csv("/kaggle/input/world-happiness-report-2026/world_happiness_2026.csv")

print(f"Shape: {df.shape}  ({df['country'].nunique()} countries, {len(df.columns)} columns)")
print(f"\nRegions: {df['region'].nunique()}")
print(f"Score range: {df['score'].min():.3f} (Afghanistan) → {df['score'].max():.3f} (Finland)")
print(f"Missing values: {df.isnull().sum().sum()}")
print()
df.head(10)

In [ ]:
df.describe().round(3)

## 2. Global Happiness Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(df['score'], bins=25, color=TEAL, edgecolor='white', linewidth=0.8, alpha=0.9)
axes[0].axvline(df['score'].mean(), color=CORAL, linewidth=2.5, linestyle='--', label=f'Mean: {df["score"].mean():.3f}')
axes[0].axvline(df['score'].median(), color=GOLD, linewidth=2.5, linestyle='--', label=f'Median: {df["score"].median():.3f}')
axes[0].set_title('Global Happiness Score Distribution', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Happiness Score (Cantril Ladder)', fontsize=11)
axes[0].set_ylabel('Number of Countries', fontsize=11)
axes[0].legend(fontsize=10)

# Colour-coded by happiness band
bands = [(7.0, 8.0, '7–8: Very Happy', '#27ae60'),
         (6.0, 7.0, '6–7: Happy', '#2ecc71'),
         (5.0, 6.0, '5–6: Moderate', '#f39c12'),
         (4.0, 5.0, '4–5: Unhappy', '#e67e22'),
         (0.0, 4.0, '<4: Very Unhappy', '#e74c3c')]

for lo, hi, label, color in bands:
    subset = df[(df['score'] >= lo) & (df['score'] < hi)]
    axes[1].barh(subset['country'], subset['score'], color=color, alpha=0.85, edgecolor='white', linewidth=0.4)

axes[1].set_title('All 147 Countries by Happiness Score', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Happiness Score', fontsize=11)
axes[1].tick_params(axis='y', labelsize=4.5)
axes[1].invert_yaxis()

legend_patches = [mpatches.Patch(color=c, label=l) for _, _, l, c in bands]
axes[1].legend(handles=legend_patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()

print(f"Very Happy (7+): {(df['score']>=7).sum()} countries ({(df['score']>=7).mean()*100:.1f}%)")
print(f"Very Unhappy (<4): {(df['score']<4).sum()} countries ({(df['score']<4).mean()*100:.1f}%)")

## 3. Regional Analysis

In [ ]:
region_stats = df.groupby('region')['score'].agg(['mean','median','min','max','count']).round(3)
region_stats.columns = ['Mean Score','Median Score','Min Score','Max Score','Countries']
region_stats = region_stats.sort_values('Mean Score', ascending=False)
print(region_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

palette = sns.color_palette("husl", df['region'].nunique())

# Box plot
region_order = df.groupby('region')['score'].median().sort_values(ascending=False).index
sns.boxplot(data=df, y='region', x='score', order=region_order, palette=palette,
            ax=axes[0], linewidth=1.2)
axes[0].set_title('Happiness Score Distribution by Region', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Happiness Score', fontsize=11)
axes[0].set_ylabel('')
axes[0].axvline(df['score'].mean(), color='red', linestyle='--', alpha=0.5, label='Global Mean')
axes[0].legend(fontsize=9)

# Mean bar
region_mean = df.groupby('region')['score'].mean().sort_values()
colors = sns.color_palette("husl", len(region_mean))
region_mean.plot.barh(ax=axes[1], color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Average Happiness Score by Region', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Mean Happiness Score', fontsize=11)
axes[1].set_ylabel('')
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{bar.get_width():.3f}', va='center', fontsize=8.5)

plt.tight_layout()
plt.show()

## 4. The Six Happiness Factors

In [ ]:
factors = ['gdp_per_capita', 'social_support', 'healthy_life_expectancy',
           'freedom', 'generosity', 'corruption']
factor_labels = ['GDP per Capita', 'Social Support', 'Healthy Life Expectancy',
                 'Freedom', 'Generosity', 'Low Corruption']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colors = [TEAL, GREEN, PURPLE, GOLD, CORAL, NAVY]

for i, (factor, label) in enumerate(zip(factors, factor_labels)):
    axes[i].scatter(df[factor], df['score'], alpha=0.6, s=35,
                    c=colors[i], edgecolors='white', linewidths=0.4)
    
    # Regression line
    z = np.polyfit(df[factor], df['score'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[factor].min(), df[factor].max(), 100)
    axes[i].plot(x_line, p(x_line), color='black', linewidth=1.8, linestyle='--', alpha=0.6)
    
    corr = df[factor].corr(df['score'])
    axes[i].set_title(f'{label}\nr = {corr:.3f}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(label, fontsize=10)
    axes[i].set_ylabel('Happiness Score' if i % 3 == 0 else '', fontsize=10)
    
    # Annotate top & bottom
    top = df.nlargest(2, factor)
    bot = df.nsmallest(2, factor)
    for _, row in pd.concat([top, bot]).iterrows():
        axes[i].annotate(row['country'], (row[factor], row['score']),
                        fontsize=6.5, alpha=0.8,
                        xytext=(4, 2), textcoords='offset points')

plt.suptitle('Happiness Score vs Each of the Six WHR Factors', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with happiness score
corr_with_score = df[factors].corrwith(df['score']).abs().sort_values(ascending=False)
corr_with_score.index = factor_labels

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(corr_with_score.index, corr_with_score.values,
               color=[TEAL, GREEN, PURPLE, GOLD, CORAL, NAVY][::-1], edgecolor='white', linewidth=0.5)
ax.set_title('Absolute Correlation of Each Factor with Happiness Score', fontsize=14, fontweight='bold')
ax.set_xlabel('|Pearson Correlation|', fontsize=11)
for bar, val in zip(bars, corr_with_score.values):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.show()

print("Factor correlations with happiness score:")
print(corr_with_score.to_string())

## 5. Correlation Matrix — All Variables

In [ ]:
all_numeric = ['score'] + factors
corr_matrix = df[all_numeric].corr()
corr_matrix.index = ['Happiness', 'GDP', 'Social Support', 'Health', 'Freedom', 'Generosity', 'Low Corruption']
corr_matrix.columns = corr_matrix.index

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, square=True, linewidths=1, linecolor='white',
            vmin=-0.5, vmax=1.0, ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 10, 'weight': 'bold'})
ax.set_title('Correlation Matrix — World Happiness 2026', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 6. Top & Bottom Countries Deep-Dive

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Top 20
top20 = df.nlargest(20, 'score').sort_values('score')
colors_top = [GREEN if s >= 7.0 else TEAL for s in top20['score']]
bars = axes[0].barh(top20['country'], top20['score'], color=colors_top, edgecolor='white', linewidth=0.5)
axes[0].set_title('🏆 Top 20 Happiest Countries (2026)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Happiness Score', fontsize=11)
axes[0].set_xlim(6.5, 8.1)
for bar, (_, row) in zip(bars, top20.iterrows()):
    axes[0].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f'#{row["rank"]}  {row["score"]:.3f}', va='center', fontsize=8.5, fontweight='bold')

# Bottom 20
bot20 = df.nsmallest(20, 'score').sort_values('score', ascending=False)
colors_bot = [CORAL if s < 3.0 else '#e67e22' for s in bot20['score']]
bars2 = axes[1].barh(bot20['country'], bot20['score'], color=colors_bot, edgecolor='white', linewidth=0.5)
axes[1].set_title('⚠️ Bottom 20 Countries (2026)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Happiness Score', fontsize=11)
axes[1].set_xlim(0, 4.5)
for bar, (_, row) in zip(bars2, bot20.iterrows()):
    axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f'#{row["rank"]}  {row["score"]:.3f}', va='center', fontsize=8.5)

plt.tight_layout()
plt.show()

In [ ]:
# Radar chart: Top 5 vs Bottom 5 factor comparison
top5 = df.nlargest(5, 'score')
bot5 = df.nsmallest(5, 'score')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, group, title, color in zip(
        axes, [top5, bot5],
        ['Top 5 Happiest Countries', 'Bottom 5 Countries'],
        [GREEN, CORAL]):
    x = np.arange(len(factor_labels))
    width = 0.12
    for i, (_, row) in enumerate(group.iterrows()):
        vals = [row[f] for f in factors]
        ax.bar(x + i*width, vals, width, label=row['country'], alpha=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(x + width*2)
    ax.set_xticklabels(factor_labels, rotation=30, ha='right', fontsize=8.5)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. GDP vs Happiness — The Classic Question

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

region_palette = dict(zip(df['region'].unique(), sns.color_palette("husl", df['region'].nunique())))
region_colors = df['region'].map(region_palette)

scatter = axes[0].scatter(df['gdp_per_capita'], df['score'],
                           c=region_colors, s=60, alpha=0.75, edgecolors='white', linewidths=0.5)
# Regression line
z = np.polyfit(df['gdp_per_capita'], df['score'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['gdp_per_capita'].min(), df['gdp_per_capita'].max(), 100)
axes[0].plot(x_line, p(x_line), color='black', linewidth=2, linestyle='--', label='Linear fit')

# Annotate key countries
for country in ['Finland', 'Afghanistan', 'United States', 'India', 'Singapore', 'Costa Rica']:
    row = df[df['country'] == country]
    if len(row):
        axes[0].annotate(country, (row['gdp_per_capita'].values[0], row['score'].values[0]),
                        fontsize=8, xytext=(5,3), textcoords='offset points', fontweight='bold')

corr_gdp = df['gdp_per_capita'].corr(df['score'])
axes[0].set_title(f'GDP per Capita vs Happiness Score\n(r = {corr_gdp:.3f})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('GDP per Capita (log scale)', fontsize=11)
axes[0].set_ylabel('Happiness Score', fontsize=11)

# Region legend
handles = [mpatches.Patch(color=c, label=r) for r, c in region_palette.items()]
axes[0].legend(handles=handles, fontsize=7, ncol=1, loc='upper left')

# Residuals from GDP
from numpy.polynomial import polynomial as P
residuals = df['score'] - p(df['gdp_per_capita'])
df['gdp_residual'] = residuals

top_overperformers = df.nlargest(10, 'gdp_residual')
bot_underperformers = df.nsmallest(10, 'gdp_residual')

all_res = pd.concat([top_overperformers, bot_underperformers]).sort_values('gdp_residual')
colors_res = [GREEN if r > 0 else CORAL for r in all_res['gdp_residual']]
axes[1].barh(all_res['country'], all_res['gdp_residual'], color=colors_res, edgecolor='white', linewidth=0.4)
axes[1].axvline(0, color='black', linewidth=1.2)
axes[1].set_title('Countries Over/Under-performing vs GDP Prediction', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual (Actual − Predicted Happiness)', fontsize=11)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()
print(f"GDP-happiness correlation: r = {corr_gdp:.3f}")
print(f"\nTop 5 overperformers (happier than GDP predicts):")
print(df.nlargest(5,'gdp_residual')[['country','score','gdp_per_capita','gdp_residual']].to_string(index=False))
print(f"\nTop 5 underperformers:")
print(df.nsmallest(5,'gdp_residual')[['country','score','gdp_per_capita','gdp_residual']].to_string(index=False))

## 8. 📱 2026 Special Focus: Social Media & Youth Wellbeing

The **2026 WHR** identifies a troubling pattern: young adults in English-speaking countries (US, UK, Canada, Australia, New Zealand) have experienced **significant declines in life satisfaction** since ~2012, which the report links to heavy social media use — a trend not observed in older cohorts or non-English-speaking nations.

Let's examine the countries highlighted in this analysis and compare their happiness trajectory indicators.

In [ ]:
# English-speaking vs comparable non-English nations
english_speaking = ['United States', 'United Kingdom', 'Canada', 'Australia', 'New Zealand', 'Ireland']
non_english_nordic = ['Finland', 'Denmark', 'Sweden', 'Norway', 'Netherlands', 'Germany', 'France']

es_data = df[df['country'].isin(english_speaking)].copy()
nen_data = df[df['country'].isin(non_english_nordic)].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Score comparison
es_scores = es_data.set_index('country')['score']
nen_scores = nen_data.set_index('country')['score']

all_comp = pd.DataFrame({
    'English-Speaking': es_scores,
    'Nordic/Western Europe': nen_scores
}).stack().reset_index()
all_comp.columns = ['country', 'group', 'score']

colors_group = {'English-Speaking': CORAL, 'Nordic/Western Europe': TEAL}
for group, group_df in all_comp.groupby('group'):
    axes[0].scatter([group]*len(group_df), group_df['score'],
                    s=80, color=colors_group[group], zorder=5, alpha=0.8, edgecolors='white', linewidths=0.5)
    for _, row in group_df.iterrows():
        axes[0].annotate(row['country'], (row['group'], row['score']),
                        fontsize=7.5, xytext=(8, 0), textcoords='offset points', alpha=0.9)
axes[0].set_title('Happiness: English-Speaking vs Nordic', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Happiness Score')

# Freedom comparison (freedom allows individual choices including social media control)
factor_compare = pd.DataFrame({
    'Factor': factor_labels * 2,
    'Score': [es_data[f].mean() for f in factors] + [nen_data[f].mean() for f in factors],
    'Group': ['English-Speaking']*6 + ['Nordic/Western Europe']*6
})

x = np.arange(len(factor_labels))
width = 0.35
bars1 = axes[1].bar(x - width/2, [es_data[f].mean() for f in factors], width,
                    label='English-Speaking', color=CORAL, alpha=0.8, edgecolor='white')
bars2 = axes[1].bar(x + width/2, [nen_data[f].mean() for f in factors], width,
                    label='Nordic/Western Europe', color=TEAL, alpha=0.8, edgecolor='white')
axes[1].set_title('Factor Comparison: English-Speaking vs Nordic', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(factor_labels, rotation=35, ha='right', fontsize=8)
axes[1].legend(fontsize=9)

# Social support vs freedom (freedom-social support gap hypothesized to relate to social media isolation)
axes[2].scatter(es_data['freedom'], es_data['social_support'], s=100, color=CORAL,
                label='English-Speaking', alpha=0.85, edgecolors='white', zorder=5)
axes[2].scatter(nen_data['freedom'], nen_data['social_support'], s=100, color=TEAL,
                label='Nordic/W. Europe', alpha=0.85, edgecolors='white', zorder=5)
for _, row in pd.concat([es_data, nen_data]).iterrows():
    axes[2].annotate(row['country'], (row['freedom'], row['social_support']),
                    fontsize=7, xytext=(4,2), textcoords='offset points', alpha=0.9)
axes[2].set_title('Freedom vs Social Support', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Freedom Score'); axes[2].set_ylabel('Social Support Score')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("📊 Average scores — English-speaking countries:")
print(es_data[['country','score','social_support','freedom']].to_string(index=False))
print(f"\n📊 Average scores — Nordic/W. European countries:")
print(nen_data[['country','score','social_support','freedom']].to_string(index=False))
print(f"\nKey gap: Nordic avg score {nen_data['score'].mean():.3f} vs English-speaking {es_data['score'].mean():.3f}")

## 9. Geospatial Happiness Map

In [ ]:
try:
    import geopandas as gpd
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    
    # Name harmonisation
    name_map = {
        'United States of America': 'United States',
        'Czech Rep.': 'Czechia',
        'S. Sudan': 'South Sudan',
        'Central African Rep.': 'Central African Republic',
        'Dem. Rep. Congo': 'DR Congo',
        'Congo': 'Congo',
        'Dominican Rep.': 'Dominican Republic',
        'Korea': 'South Korea',
        "Côte d'Ivoire": 'Ivory Coast',
        'Bosnia and Herz.': 'Bosnia and Herzegovina',
        'Macedonia': 'North Macedonia',
    }
    world['name'] = world['name'].replace(name_map)
    merged = world.merge(df, left_on='name', right_on='country', how='left')
    
    fig, ax = plt.subplots(figsize=(18, 9))
    world.plot(ax=ax, color='#f0f0f0', edgecolor='white', linewidth=0.4)
    merged.dropna(subset=['score']).plot(
        column='score', ax=ax, cmap='RdYlGn',
        vmin=1, vmax=8,
        legend=True,
        legend_kwds={'label': 'Happiness Score (Cantril Ladder)', 'orientation': 'horizontal',
                     'shrink': 0.5, 'pad': 0.02},
        edgecolor='white', linewidth=0.3
    )
    ax.set_title('World Happiness Report 2026 — Global Happiness Map', fontsize=16, fontweight='bold', pad=15)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print("✅ Geospatial map rendered")
except ImportError:
    print("geopandas not available — showing regional bar chart instead")
    region_scores = df.groupby('region')['score'].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    region_scores.plot.bar(ax=ax, color=sns.color_palette("RdYlGn", len(region_scores)), edgecolor='white')
    ax.set_title('Average Happiness Score by World Region — WHR 2026', fontsize=14, fontweight='bold')
    ax.set_ylabel('Average Happiness Score'); ax.tick_params(axis='x', rotation=40)
    plt.tight_layout(); plt.show()

## 10. Regression Model — Predicting Happiness Score

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

X = df[factors].values
y = df['score'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42),
}

print("5-Fold Cross-Validation Results:\n")
print(f"{'Model':<25} {'Mean R²':>10} {'Std R²':>10} {'Mean MAE':>10}")
print("-" * 58)

results = {}
for name, model in models.items():
    X_input = X_scaled if 'Regression' in name else X
    r2_scores = cross_val_score(model, X_input, y, cv=kf, scoring='r2')
    mae_scores = -cross_val_score(model, X_input, y, cv=kf, scoring='neg_mean_absolute_error')
    results[name] = {'r2': r2_scores.mean(), 'r2_std': r2_scores.std(), 'mae': mae_scores.mean()}
    print(f"{name:<25} {r2_scores.mean():>10.4f} {r2_scores.std():>10.4f} {mae_scores.mean():>10.4f}")

In [ ]:
# Fit best model and inspect
best_model = GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42)
best_model.fit(X, y)
y_pred = best_model.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predicted vs Actual
axes[0].scatter(y, y_pred, alpha=0.65, s=50, c=TEAL, edgecolors='white', linewidths=0.4)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'k--', linewidth=1.5, label='Perfect fit')
residuals = y - y_pred
for i, row in df.iterrows():
    if abs(residuals[i]) > 0.5:
        axes[0].annotate(row['country'], (y[i], y_pred[i]), fontsize=7, alpha=0.8,
                        xytext=(4, 2), textcoords='offset points')
axes[0].set_title(f'Predicted vs Actual Happiness Score\nR² = {r2_score(y, y_pred):.4f}  |  MAE = {mean_absolute_error(y, y_pred):.4f}',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Score'); axes[0].set_ylabel('Predicted Score')
axes[0].legend()

# Feature importance
fi = pd.Series(best_model.feature_importances_, index=factor_labels).sort_values()
fi.plot.barh(ax=axes[1], color=[TEAL, GREEN, PURPLE, GOLD, CORAL, NAVY][:len(fi)], edgecolor='white')
axes[1].set_title('Feature Importance (Gradient Boosting)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Relative Importance')
for bar, val in zip(axes[1].patches, fi.values):
    axes[1].text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 📋 Key Findings from the 2026 World Happiness Report

### 🥇 Rankings
- **Finland** holds the top spot for the **8th consecutive year** (score: 7.764)
- **Costa Rica** is the biggest surprise at #4, punching far above its GDP
- **Afghanistan** remains last at 1.446 — a full 6.3 points below Finland

### 🌍 Regional Patterns
- **Western Europe** and **North America + ANZ** dominate the top 20
- **Sub-Saharan Africa** and **South Asia** occupy the bottom of the rankings
- **Latin America** consistently outperforms its income levels (the "Latin American paradox")

### 📊 Factor Insights
- **Social support** and **GDP per capita** are the two strongest predictors (r > 0.75)
- **Generosity** shows the weakest correlation — wealth ≠ generosity
- **Low corruption** matters more in the top tier than the bottom tier

### 📱 2026 Special Focus: Social Media & Youth
- English-speaking nations (US, UK, Canada, Australia, NZ) show a **notable gap** vs Nordic peers
- Despite similar or higher GDP, they score 0.3–1.0 points lower than comparable Nordics
- The 2026 report links this to **social media adoption and youth wellbeing declines** since ~2012
- Costa Rica's high ranking without high social media penetration offers a counterpoint

### 🔮 Model Performance
- Gradient Boosting achieves **R² > 0.95** — the six factors explain >95% of happiness variance
- Largest residuals are Lebanon (crisis) and Costa Rica (paradox) — real-world outliers

---
*Data source: World Happiness Report 2026 — Gallup World Poll (2023–2025 averages)*  
*If this notebook was useful, please upvote! 🙏*